In [2]:
from multiprocessing import set_start_method
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# **Must** happen before torch or vllm ever touches CUDA
set_start_method("spawn", force=True)


from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerFast, PreTrainedModel
from transformers import Trainer, TrainingArguments
from trl import SFTConfig, SFTTrainer
from trl import setup_chat_format
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import load_dataset, Dataset
from concurrent.futures import ThreadPoolExecutor
from trl import DataCollatorForCompletionOnlyLM
import torch
from vllm import LLM, SamplingParams
from vllm.model_executor import set_random_seed as vllm_set_random_seed
from drgrpo_grader import r1_zero_reward_fn
import gc
from unittest.mock import patch
import wandb
import safetensors
import os
import json
import numpy as np
import time
import random 
import operator

model_name = "Qwen/Qwen2.5-1.5B"
tokenizer_dir =  "checkpoint/sft/tokenizer"

tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, local_files_only=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     ).to(device)



ds = load_dataset("openai/gsm8k", "main")

def preprocess_dataset(ds, usage, data_sampling_ratio=1, seed=125):
    sample_cnt = int(len(ds[usage]["answer"]) * data_sampling_ratio)
    random.seed(seed)
    sampled_data_idx = random.sample(range(0, len(ds[usage]["answer"])), sample_cnt)
    getter = operator.itemgetter(*sampled_data_idx)
    questions, answers =  getter(ds[usage]["question"]), getter(ds[usage]["answer"])
    print(len(questions), len(answers))
    with open("prompts/r1_zero.prompt", "r", encoding="utf-8") as f:
        prompt_string = f.read()

    def process_question(q):
        return prompt_string.format(question=q)
    def process_ground_truth(ans):
        return ans.split('\n#### ')[1]
    def process_prompt_completion(q, ans):
        prompt = prompt_string.format(question=q)
        cot =' ' + ans.split('\n#### ')[0] + ' </think>'
        gt = f" <answer> {ans.split('\n#### ')[1]} </answer>"
        return prompt + cot + gt
    with ThreadPoolExecutor() as executor:
        question_prompts = list(executor.map(process_question, ds[usage]["question"]))
    with ThreadPoolExecutor() as executor:
        ground_truth = list(executor.map(process_ground_truth, ds[usage]["answer"]))
    with ThreadPoolExecutor() as executor:
        prompt_completion = list(executor.map(process_prompt_completion, ds[usage]["question"], ds[usage]["answer"]))
    return question_prompts, ground_truth, prompt_completion

training_question_prompt, training_gt, training_data = preprocess_dataset(ds, 'train', 0.4)
test_prompt, test_gt =  preprocess_dataset(ds, 'test')[0], preprocess_dataset(ds, 'test')[1]
train_ds = Dataset.from_dict({
    "text":training_data,
    "prompt":  training_question_prompt,
    "gt":training_gt
    })
val_ds = Dataset.from_dict({
    "prompt": test_prompt[:len(test_prompt)//2],
    "gt": test_gt[:len(test_gt)//2],

})
test_ds = Dataset.from_dict({
    "prompt": test_prompt[len(test_prompt)//2:],
    "gt": test_gt[len(test_gt)//2:],

})
# val_ds = Dataset.from_dict({
#     "prompt": test_prompt,
#     "gt": test_gt,

# })


# Build a collator whose response_template matches your prompt ending

collator = DataCollatorForCompletionOnlyLM(
    tokenizer = tokenizer,
    # Anything before *and including* this string gets label = -100
    response_template  = r"Assistant: <think>",   # note the space after >
    
)



INFO 06-24 07:34:59 [__init__.py:243] Automatically detected platform cuda.
2989 2989
1319 1319
1319 1319


In [3]:
def init_vllm(hf_model_dir: str,  device: str, seed: int, gpu_memory_utilization: float = 0.85):
    """Start the inference process, here we use vLLM to hold a model on
    a GPU separate from the policy.
    """
    vllm_set_random_seed(seed)
   
    # Monkeypatch from TRL:
    # https://github.com/huggingface/trl/blob/
    # 22759c820867c8659d00082ba8cf004e963873c1/trl/trainer/grpo_trainer.py
    # Patch vLLM to make sure we can
    # (1) place the vLLM model on the desired device (world_size_patch) and
    # (2) avoid a test that is not designed for our setting (profiling_patch).
    world_size_patch = patch("torch.distributed.get_world_size", return_value=1)
    profiling_patch = patch(
    "vllm.worker.worker.Worker._assert_memory_footprint_increased_during_profiling",
    return_value=None
    )
    with world_size_patch, profiling_patch:
        return LLM(
        #model="Qwen/Qwen2.5-1.5B",    # base model dir or HF name
        #peft_model="./checkpoint/sft/sft_lora_results3/checkpoint-1500",  # path to LoRA adapter
        model=hf_model_dir,
        #tokenizer=tokenizer,
        #tokenizer_mode='auto',
        device=device,
        dtype=torch.bfloat16,
        enable_prefix_caching=True,
        gpu_memory_utilization=gpu_memory_utilization,
        
        )

def data_filter(vllm, model_dir, ds, G=8, sampling_temperature=1,top_p=1.0,sampling_max_tokens=1024,sampling_min_tokens=4, seed=233):
   
    ds_prompt = []
    ds_gt = []
    ds_prompt_completion = []

    
    start_time = time.time()
    sampling_params = SamplingParams(
        temperature=sampling_temperature,
        top_p=top_p,
        max_tokens=sampling_max_tokens,
        min_tokens=sampling_min_tokens,
        n=G,
        seed=seed,
        stop=["</answer>"]
    )
    sampling_params.include_stop_str_in_output = True
    all_prompt_texts = ds['prompt']
    all_answer_gt = ds['gt']
    all_prompt_completion = ds['text']
    all_model_outputs = vllm.generate(all_prompt_texts, sampling_params)
    start2_time = time.time()
    print(f'generate all response time: {start2_time - start_time}------------')

    for i, (outputs, gt) in enumerate(zip(all_model_outputs, all_answer_gt)):
        prompt = outputs.prompt
        for output in outputs.outputs:
            
            generated_text = output.text
            res = r1_zero_reward_fn(generated_text, gt)
            if res['reward'] == 1:
                ds_prompt.append(prompt)
                ds_gt.append(gt)
                ds_prompt_completion.append(prompt+generated_text)



    filtered_ds = Dataset.from_dict({
        "text":ds_prompt_completion,
        "prompt":  ds_prompt,
        "gt":ds_gt
    })
    return filtered_ds




   


In [6]:
initial_sft_model_dir = "./checkpoint/sft/sft_lora_results3/my_merged_model"
vllm = init_vllm(initial_sft_model_dir, device, 233)
training_question_prompt, training_gt, training_data = preprocess_dataset(ds, 'train', 0.4, seed=21)

train_ds = Dataset.from_dict({
    "text":training_data,
    "prompt_temp":  training_question_prompt,
    "gt_temp":training_gt
    })

filtered_train_ds = data_filter(initial_sft_model_dir, vllm, train_ds, G=4, sampling_temperature=1,top_p=1.0,sampling_max_tokens=1024,sampling_min_tokens=4, seed=233)

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     ).to(device)

INFO 06-24 07:39:21 [config.py:793] This model supports multiple tasks: {'embed', 'reward', 'generate', 'score', 'classify'}. Defaulting to 'generate'.
INFO 06-24 07:39:21 [config.py:2118] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-24 07:39:26 [__init__.py:243] Automatically detected platform cuda.
INFO 06-24 07:39:29 [core.py:438] Waiting for init message from front-end.
INFO 06-24 07:39:29 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 06-24 07:39:29 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 06-24 07:39:29 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 06-24 07:39:29 [core.py:65] Initializing a V1 LLM engine (v0.9.0.1) with config: model='./checkpoint/sft/sft_lora_results3/my_merged_model', speculative_config=None, tokenizer='./checkpoint/sft/sft_lora_results3/my_merged_model', s

Process EngineCore_0:
Traceback (most recent call last):
  File "/home/sagemaker-user/.conda/envs/myenv3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/sagemaker-user/.conda/envs/myenv3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/sagemaker-user/.conda/envs/myenv3/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 504, in run_engine_core
    raise e
  File "/home/sagemaker-user/.conda/envs/myenv3/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 491, in run_engine_core
    engine_core = EngineCoreProc(*args, **kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sagemaker-user/.conda/envs/myenv3/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 390, in __init__
    super().__init__(vllm_config, executor_class, log_stats,
  File "/home/sagemaker-user/.conda/envs/myenv3/lib/python3.12/site-packages/vllm/v1/engine/core.p

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [11]:
filtered_train_ds['text'][0]

'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAssistant: <think> Natalia sold clips to 48 of her friends in April <br> In May Natalia only sold half as much clips which is 48/2 = 24 clips <br> Thus, Natalia had 48 clips + 24 clips = 72 clips in April and May </think> <answer> 72 </answer>'

### training

In [7]:
from peft import PeftModel
import numpy

#model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")



sft_config = SFTConfig(
    max_seq_length=1024,
    output_dir="./checkpoint/expert_iteration/sft_iter1",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-6,  
    num_train_epochs=1,
    logging_steps=10,
    #dataset_text_field="text",
    label_names=["labels"],
    warmup_ratio=0.05,
    report_to = "wandb",  
    bf16=True,   
   
    #pad_token_id=eos_id,
    #eos_token_id=eos_id,          # <— this is what TRL will use to stop
    # you can also set other generation defaults here if you like
)


# # LoRA 配置
# lora_config = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     inference_mode=False,
#     r=32,  
#     lora_alpha=32,
#     lora_dropout=0.1,
   
#     modules_to_save=["embed_tokens", "lm_head"],
#     target_modules='all-linear'
#     #target_modules = ["q_proj","v_proj"]
# )

# # 将LoRA配置应用到模型
# peft_model = get_peft_model(model, lora_config)

# # use following when continue training
# #peft_model = PeftModel.from_pretrained(model, "./checkpoint/sft/sft_lora_results3/checkpoint-500", is_trainable=True ) 




# # 训练参数配置
# training_args = TrainingArguments(
#     output_dir="./checkpoint/sft/sft_lora_results",
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     learning_rate=1e-4,  
#     num_train_epochs=3,
#     fp16=True,  
#     logging_steps=1
# )

# 使用Trainer API进行训练
trainer = SFTTrainer(
    model=initial_sft_model_dir,
    train_dataset=filtered_train_ds,
    data_collator=collator, 
    args=sft_config,
   
    
    #data_collator=torch.utils.data.DataCollatorWithPadding(tokenizer=tokenizer)
)

#with torch.serialization.safe_globals([numpy.core.multiarray._reconstruct]):


trainer.train(resume_from_checkpoint=True)  #set to false when not continue training


Converting train dataset to ChatML:   0%|          | 0/7457 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/7457 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7457 [00:00<?, ? examples/s]

KeyError: 'completion'

### Evaluation

In [ ]:
def sft_evaluation(hf_policy_dir: str, vllm,  val_ds,  device: str, out_dir):
    start_time = time.time()
    #format_reward, answer_reward, answer = [], [], []
    # total_response_len = 0
    # total_response_len_correct = 0
    # total_response_len_incorrect = 0
    # total_sample = 0
    # total_correct_sample = 0
    # total_incorrect_sample = 0
    response_avg_entropy_lst = [] #(batch,)
    response_len_lst = [] #(batch,)
    correct_lst = []
    incorrect_lst = []
    #initialize llm for vllm
    #llm = init_vllm(hf_policy_dir, device, 233)
    #load_policy_into_vllm_instance(policy, tokenizer, llm)
    sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["</answer>"]
    )
    sampling_params.include_stop_str_in_output = True
    all_prompt_texts = val_ds['prompt']
    all_answer_gt = val_ds['gt']
    # for batch in val_dataset:
    #     all_prompt_texts.extend(batch['prompt_texts'])
    #     all_answer_gt.extend(batch['answer_gt'] ) 
   
    all_model_outputs = vllm.generate(all_prompt_texts, sampling_params)
    start2_time = time.time()
    print(f'generate all response time: {start2_time - start_time}------------')

    # still need policy model for eval mode
    #policy.eval()
    os.makedirs(out_dir.rsplit('/', 1)[0], exist_ok=True) 
    with open(out_dir, "a", encoding="utf-8") as f:
        format_reward, answer_reward, reward = 0, 0, 0
     
        for i, (output, gt) in enumerate(zip(all_model_outputs, all_answer_gt)):
            prompt = output.prompt
            generated_text = output.outputs[0].text
            
            res = r1_zero_reward_fn(generated_text, gt)
            # format_reward+= res['format_reward']
            # answer_reward+= res['answer_reward']
            # reward+= res['reward']
            #print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}, format_reward: {str(res['format_reward'])}, answer_reward: {str(res['answer_reward'])}, reward: {str(res['reward'])}")
            #final_output.append([res['format_reward'], res['answer_reward'], res['reward']])
            dp = {
                "prompt": f"{prompt}",
                "ground_truth": gt, 
                "output": f"{generated_text}",
                "format_reward": res['format_reward'],
                "answer_reward": res['answer_reward'],
                "reward": res['reward'],
                #"avg_response_token_entropy": response_avg_entropy_lst[i]
            }
            correct_lst.append(int(res['reward']==1))
            incorrect_lst.append(int(res['reward']!=1))
            format_reward+= res['format_reward']
            answer_reward+= res['answer_reward']
            reward+= res['reward']

            json.dump(dp, f, ensure_ascii=False)
            
    #total_response_len_correct = np.sum(np.array(correct_lst) * np.array(response_len_lst))
    #total_response_len_incorrect = np.sum(np.array(incorrect_lst) * np.array(response_len_lst))
    total_correct_sample = np.sum(np.array(correct_lst))
    total_incorrect_sample = np.sum(np.array(incorrect_lst))
    total_sample = len(all_model_outputs)
    #total_response_len = np.sum(np.array(response_len_lst))

    print(f'total_correct_sample: {total_correct_sample}, total_incorrect_sample: {total_incorrect_sample}')
    print(format_reward, answer_reward, reward)
    #print(f'final stats\navg_response_len={total_response_len/total_sample:.2f}, avg_response_len_correct={total_response_len_correct/total_correct_sample:.2f}, avg_response_len_incorrect={total_response_len_incorrect/total_incorrect_sample:.2f}')
    #print("some check:", total_response_len_correct+total_response_len_incorrect==total_response_len)
    #print("moer check:", total_correct_sample+total_incorrect_sample==total_sample)
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    return total_correct_sample / total_sample